# Exercise 06 — RESTful Web Services: the Server
### 🏗️ Building the SmartCity Sensor Catalog

In Exercise 05 you were the **client** consuming an API.
Now you are the **server** — you will build a RESTful sensor catalog from scratch.

This is exactly what your lecture describes:
> *"A device will implement a webserver [...] through webservices the device can be configured,
> send data/commands, receive data/commands"*

By the end of this exercise you will have a working sensor registry API that:
- Stores sensors and their readings
- Validates incoming data (using what you learnt in Exercise 02!)
- Returns proper HTTP status codes
- Exposes a filtered query endpoint

---

## 1️⃣  Flask basics — your first endpoint

A Flask **route** maps a URL pattern + HTTP method to a Python function.

In [ ]:
# ── Understand the anatomy of a Flask route ────────────────────────────────
# (Read-only cell — do not run, just study the structure)

from flask import Flask, jsonify, request

app = Flask(__name__)

@app.route("/sensors", methods=["GET"])   # URL path + allowed methods
def get_sensors():
    data = [{"id": "TRN-001"}]            # build the response
    return jsonify(data), 200             # return JSON + status code

# The decorator @app.route() is what connects:
#   URL:    /sensors
#   Method: GET
#   Handler: the function below it

print("Route anatomy understood ✓")

## 2️⃣  Design the API first (URL schema)

Good REST design names URLs after **resources** (nouns), not actions (verbs).

```
GET    /sensors              → list all sensors
GET    /sensors/<id>         → get one sensor
POST   /sensors              → register a new sensor
PUT    /sensors/<id>         → update sensor reading
DELETE /sensors/<id>         → deregister a sensor

GET    /sensors?alert=true   → filter: only sensors with PM2.5 > 20
GET    /sensors/<id>/history → reading history for one sensor
POST   /sensors/<id>/history → append a new reading to history
```

## 3️⃣  Build the full server

We will build this step by step, then start it.

In [ ]:
import threading, time, json, logging
from flask import Flask, jsonify, request
from jsonschema import validate, ValidationError

# ── Schema (reused from Exercise 02) ──────────────────────────────────────
SENSOR_SCHEMA = {
    "type": "object",
    "required": ["sensor_id", "location", "temperature_c", "pm25_ugm3"],
    "properties": {
        "sensor_id":    {"type": "string", "pattern": "^TRN-[0-9]{3}$"},
        "location":     {"type": "string", "minLength": 2},
        "temperature_c":{"type": "number", "minimum": -40, "maximum": 85},
        "humidity_pct": {"type": "number", "minimum": 0,   "maximum": 100},
        "pm25_ugm3":    {"type": "number", "minimum": 0},
    }
}

# ── In-memory storage ─────────────────────────────────────────────────────
SENSORS  = {}    # sensor_id → sensor dict
HISTORY  = {}    # sensor_id → list of reading dicts

def seed():
    """Pre-load the 5 Turin sensors."""
    initial = [
        {"sensor_id": "TRN-001", "location": "Piazza Castello",  "temperature_c": 22.4, "pm25_ugm3": 12.1, "humidity_pct": 58},
        {"sensor_id": "TRN-002", "location": "Piazza Vittorio",  "temperature_c": 23.1, "pm25_ugm3": 18.4, "humidity_pct": 61},
        {"sensor_id": "TRN-003", "location": "Mirafiori",        "temperature_c": 21.0, "pm25_ugm3": 35.2, "humidity_pct": 70},
        {"sensor_id": "TRN-004", "location": "Lingotto",         "temperature_c": 24.5, "pm25_ugm3": 22.7, "humidity_pct": 52},
        {"sensor_id": "TRN-005", "location": "Parco del Po",     "temperature_c": 20.8, "pm25_ugm3":  8.3, "humidity_pct": 75},
    ]
    for s in initial:
        SENSORS[s["sensor_id"]] = s
        HISTORY[s["sensor_id"]] = []

seed()

# ── Flask app ──────────────────────────────────────────────────────────────
app2 = Flask("catalog")
logging.getLogger('werkzeug').setLevel(logging.ERROR)

# ── Helper ─────────────────────────────────────────────────────────────────
def validate_body(body):
    """Returns (True, None) or (False, error_message)."""
    try:
        validate(instance=body, schema=SENSOR_SCHEMA)
        return True, None
    except ValidationError as e:
        return False, e.message

# ── Routes ─────────────────────────────────────────────────────────────────

@app2.route("/sensors", methods=["GET"])
def get_all():
    sensors = list(SENSORS.values())
    # Optional filter: /sensors?alert=true
    if request.args.get("alert") == "true":
        sensors = [s for s in sensors if s["pm25_ugm3"] > 20]
    return jsonify(sensors), 200

@app2.route("/sensors/<sensor_id>", methods=["GET"])
def get_one(sensor_id):
    if sensor_id not in SENSORS:
        return jsonify({"error": f"{sensor_id} not found"}), 404
    return jsonify(SENSORS[sensor_id]), 200

@app2.route("/sensors", methods=["POST"])
def create_sensor():
    body = request.get_json(silent=True)
    if not body:
        return jsonify({"error": "request body must be JSON"}), 400
    ok, err = validate_body(body)
    if not ok:
        return jsonify({"error": err}), 400
    sid = body["sensor_id"]
    if sid in SENSORS:
        return jsonify({"error": f"{sid} already exists, use PUT to update"}), 409
    SENSORS[sid] = body
    HISTORY[sid] = []
    return jsonify(body), 201

@app2.route("/sensors/<sensor_id>", methods=["PUT"])
def update_sensor(sensor_id):
    if sensor_id not in SENSORS:
        return jsonify({"error": f"{sensor_id} not found"}), 404
    updates = request.get_json(silent=True) or {}
    SENSORS[sensor_id].update(updates)
    return jsonify(SENSORS[sensor_id]), 200

@app2.route("/sensors/<sensor_id>", methods=["DELETE"])
def delete_sensor(sensor_id):
    if sensor_id not in SENSORS:
        return jsonify({"error": f"{sensor_id} not found"}), 404
    SENSORS.pop(sensor_id)
    HISTORY.pop(sensor_id, None)
    return jsonify({"deleted": sensor_id}), 200

@app2.route("/sensors/<sensor_id>/history", methods=["GET"])
def get_history(sensor_id):
    if sensor_id not in SENSORS:
        return jsonify({"error": f"{sensor_id} not found"}), 404
    return jsonify(HISTORY.get(sensor_id, [])), 200

@app2.route("/sensors/<sensor_id>/history", methods=["POST"])
def add_reading(sensor_id):
    if sensor_id not in SENSORS:
        return jsonify({"error": f"{sensor_id} not found"}), 404
    reading = request.get_json(silent=True)
    if not reading or "temperature_c" not in reading:
        return jsonify({"error": "reading must include temperature_c"}), 400
    reading["timestamp"] = time.time()
    HISTORY[sensor_id].append(reading)
    # Also update the current reading on the sensor
    SENSORS[sensor_id].update({k: v for k, v in reading.items() if k != "timestamp"})
    return jsonify(reading), 201

# Start server
t2 = threading.Thread(target=lambda: app2.run(port=5001, use_reloader=False))
t2.daemon = True
t2.start()
time.sleep(1)
print("🚀 SmartCity Catalog API running at http://localhost:5001")

## 4️⃣  Test your server with the client

Now use `requests` (from Exercise 05) to verify every endpoint works.

In [ ]:
import requests
BASE = "http://localhost:5001"

# --- GET all ---
r = requests.get(f"{BASE}/sensors")
print(f"GET /sensors → {r.status_code}, {len(r.json())} sensors")

# --- GET filtered (alert) ---
r = requests.get(f"{BASE}/sensors", params={"alert": "true"})
print(f"GET /sensors?alert=true → {r.status_code}, {len(r.json())} sensors in alert")

# --- POST valid ---
r = requests.post(f"{BASE}/sensors", json={
    "sensor_id": "TRN-006", "location": "Porta Nuova",
    "temperature_c": 21.5, "pm25_ugm3": 14.0, "humidity_pct": 60
})
print(f"POST /sensors (valid) → {r.status_code}")

# --- POST invalid (schema violation) ---
r = requests.post(f"{BASE}/sensors", json={
    "sensor_id": "TRN-007", "location": "X",   # location too short!
    "temperature_c": 20.0, "pm25_ugm3": 10.0
})
print(f"POST /sensors (invalid) → {r.status_code}: {r.json()['error'][:60]}")

# --- POST duplicate ---
r = requests.post(f"{BASE}/sensors", json={
    "sensor_id": "TRN-001", "location": "Piazza Castello",
    "temperature_c": 22.0, "pm25_ugm3": 12.0
})
print(f"POST /sensors (duplicate) → {r.status_code}: {r.json()['error']}")

## 5️⃣  Simulate a sensor pushing readings over time

In [ ]:
import random, time
random.seed(7)

print("Simulating 5 readings from TRN-001...")
temp = 22.0
for i in range(5):
    temp += random.uniform(-0.5, 0.8)
    pm25 = round(random.uniform(10, 30), 1)
    reading = {"temperature_c": round(temp, 1), "pm25_ugm3": pm25}
    r = requests.post(f"{BASE}/sensors/TRN-001/history", json=reading)
    print(f"  Reading {i+1}: {reading} → {r.status_code}")
    time.sleep(0.2)

# Fetch the history
history = requests.get(f"{BASE}/sensors/TRN-001/history").json()
print(f"\nHistory length: {len(history)} readings")
temps = [h["temperature_c"] for h in history]
print(f"Temp range: {min(temps):.1f} – {max(temps):.1f} °C")

---
## 🏆 Challenges

1. Add a `GET /sensors/<id>/history?last=N` query parameter that returns only the last N readings.
2. Add a `GET /stats` endpoint that returns `{count, avg_temp, max_pm25, alert_count}` across all sensors.
3. Add a `GET /sensors/<id>/status` endpoint that returns `{"status": "ok"}` or `{"status": "alert"}` based on PM2.5.
4. Open your browser and visit `http://localhost:5001/sensors` — you are using a web browser as a REST client!

In [ ]:
# Your code here
